# Análisis XAI del modelo de segmentación de isquemia

Este cuaderno acompaña la memoria del TFG **Herramienta de análisis de imágenes biomédicas**, desarrollado en colaboración con **ICTS BioImagen Complutense**. Su finalidad es documentar y ejecutar un análisis de explicabilidad del modelo de segmentación binaria de lesión isquémica en resonancia magnética preclínica de ratón.

El análisis se plantea sobre la tubería actual del proyecto: primero se obtiene la región cerebral y después se evalúa el modelo de isquemia sobre la imagen enmascarada cuando `USE_BRAIN_MASK = True`. Esta decisión debe mantenerse explícita, ya que afecta tanto a la predicción como a la interpretación de los mapas de atribución.

Métodos incluidos: Grad-CAM, Grad-CAM++, Integrated Gradients, sensibilidad por oclusión, curvas de Deletion/Insertion, curvas MoRF/LeRF y AOPC.

Flujo general del análisis:

1. Configurar rutas, umbrales y parámetros de XAI.
2. Cargar funciones auxiliares de lectura, preprocesado, inferencia, XAI y métricas.
3. Resolver automáticamente la capa objetivo del modelo de isquemia.
4. Ejecutar pruebas rápidas sobre un corte con lesión y otro sin lesión.
5. Procesar todos los cortes de todos los volúmenes `CASE_*`.
6. Exportar métricas por corte, por método y por punto de curva.
7. Seleccionar ejemplos cualitativos representativos.
8. Guardar tablas y montajes listos para revisión e inclusión en la memoria.

Criterios metodológicos principales:

- La tarea es segmentación binaria, no clasificación de cortes.
- El objetivo escalar de XAI se define a partir de la probabilidad media de lesión dentro de la máscara de lesión predicha.
- Las métricas de fidelidad se calculan únicamente cuando existe una región predicha que pueda explicarse.
- Los cortes se agrupan en cortes con lesión manual, sin lesión manual y cortes extremos del volumen.
- Las explicaciones se interpretan como evidencia de coherencia espacial y apoyo al análisis del modelo, no como prueba causal ni validación clínica independiente.


## 1. Configuración del experimento

Esta celda concentra los parámetros editables del cuaderno: rutas del conjunto de datos y de los modelos, uso de máscara cerebral, umbrales de segmentación, capa objetivo y resolución de los métodos de perturbación.

Para que los resultados sean trazables en la memoria, conviene registrar cualquier cambio de modalidad, modelo, umbral o preprocesado. Las salidas se guardan en una carpeta con marca temporal dentro del directorio del conjunto de datos.


In [ ]:
from pathlib import Path
from datetime import datetime

DATASET_DIR = "test_dataset"
MODEL_ISQ_PATH = "models/isquemia_unet_model.h5"
MODEL_BRAIN_PATH = "models/brain_unet_model.h5"

USE_BRAIN_MASK = True
IMG_TARGET = 120
BRAIN_THRESHOLD = 0.5
ISQ_THRESHOLD = 0.8

TARGET_LAYER_NAME = None
SCORE_MODE = "pred_mask_mean_probability"
ISQ_MODEL_OUTPUTS_LOGITS = None  # None = infer from the final activation; True = apply sigmoid once; False = already probabilities.
PERTURBATION_BASELINE = "mean"  # "mean" is the default for quantitative curves; "black" can be tested for sensitivity.


IG_STEPS = 64
CURVE_STEPS = 20
OCCLUSION_PATCH = 12
OCCLUSION_STRIDE = 6
SELECTED_SLICES_PER_GROUP = 6

ALLOW_TOPK_FALLBACK_FOR_VISUALIZATION = False
TOPK_FALLBACK_FRACTION = 0.01
TOPK_FALLBACK_MIN_PIXELS = 32

# Memory-oriented defaults
SCORING_BATCH_SIZE = 8
GRADCAMPP_MEMORY_SAFE = True
SMOKE_TEST_IG_STEPS = 16
SMOKE_TEST_CURVE_STEPS = 6
SMOKE_TEST_OCCLUSION_PATCH = 24
SMOKE_TEST_OCCLUSION_STRIDE = 24
SMOKE_TEST_CURVE_METHODS = ("gradcam",)

OPTIONAL_INSTALL_MISSING_DEPENDENCIES = False

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = Path(DATASET_DIR) / "_xai_isquemia_analysis" / timestamp
CSV_DIR = OUTPUT_ROOT / "csv"
FIGURES_DIR = OUTPUT_ROOT / "figures"

CSV_DIR.mkdir(parents=True, exist_ok=True)
for group_name in ("lesion", "non_lesion", "edge_first_last3"):
    (FIGURES_DIR / group_name).mkdir(parents=True, exist_ok=True)

print("Configuration ready")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"MODEL_ISQ_PATH: {MODEL_ISQ_PATH}")
print(f"MODEL_BRAIN_PATH: {MODEL_BRAIN_PATH}")
print(f"OUTPUT_ROOT: {OUTPUT_ROOT}")
print(f"SCORING_BATCH_SIZE: {SCORING_BATCH_SIZE}")
print(f"GRADCAMPP_MEMORY_SAFE: {GRADCAMPP_MEMORY_SAFE}")
print(f"SCORE_MODE: {SCORE_MODE}")
print(f"PERTURBATION_BASELINE: {PERTURBATION_BASELINE}")


## 2. Comprobación del entorno

Se comprueba la disponibilidad de las dependencias principales antes de continuar. Esta celda ayuda a reproducir el análisis en otro equipo o en un entorno de notebook, y deja impresas las versiones importadas.


In [ ]:
import importlib
import platform
import subprocess
import sys

REQUIRED_PACKAGES = {
    "tensorflow": "tensorflow",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "PIL": "Pillow",
    "nibabel": "nibabel",
}

missing_packages = []
imported_versions = {}

for module_name, pip_name in REQUIRED_PACKAGES.items():
    try:
        module = importlib.import_module(module_name)
        imported_versions[module_name] = getattr(module, "__version__", "unknown")
    except Exception:
        missing_packages.append((module_name, pip_name))

print(f"Python: {platform.python_version()}")
if imported_versions:
    print("Imported package versions:")
    for module_name, version in imported_versions.items():
        print(f"  - {module_name}: {version}")
else:
    print("No required packages are currently importable in this environment.")

if missing_packages:
    print("Missing packages detected:")
    for module_name, pip_name in missing_packages:
        print(f"  - {module_name} (pip install {pip_name})")
else:
    print("All required packages are available.")

if missing_packages and OPTIONAL_INSTALL_MISSING_DEPENDENCIES:
    install_targets = [pip_name for _, pip_name in missing_packages]
    print(f"Installing missing packages: {install_targets}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *install_targets])
    print("Installation complete. Re-run this cell, then continue.")
elif missing_packages:
    print("Set OPTIONAL_INSTALL_MISSING_DEPENDENCIES = True and re-run this cell to install automatically, or install manually.")


## 3. Importación de librerías

El cuaderno prepara las librerías necesarias para cargar volúmenes NIfTI, manipular máscaras 2D, ejecutar los modelos TensorFlow/Keras y generar las figuras y tablas finales.


In [ ]:
if missing_packages:
    missing_names = ", ".join(module_name for module_name, _ in missing_packages)
    raise ImportError(
        "Cannot continue because the following packages are missing: "
        f"{missing_names}. Install them first or enable OPTIONAL_INSTALL_MISSING_DEPENDENCIES."
    )

import gc
import math
import os
import re
import warnings
from collections import defaultdict

import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from PIL import Image, ImageFilter
from tensorflow.keras import Model
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.models import load_model

np.random.seed(42)
tf.random.set_seed(42)
warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["image.cmap"] = "gray"

print(f"TensorFlow version: {tf.__version__}")


## 4. Funciones auxiliares de segmentación y preprocesado

Este bloque define las utilidades de carga, normalización, redimensionado, unión de máscaras y ejecución de la tubería de inferencia. Cuando existen varias máscaras de lesión asociadas al mismo corte, se combinan mediante unión antes de calcular métricas o explicaciones.

El redimensionado y el uso de la máscara cerebral se mantienen explícitos porque pueden influir en resultados cuantitativos y en la interpretación de las atribuciones.


In [ ]:
# ============================================================
# Helper functions: losses, I/O, preprocessing, inference
# ============================================================

def dice_coef(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
    )

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

def dice_numpy(y_true, y_pred, smooth=1e-6):
    y_true = y_true.astype(np.uint8)
    y_pred = y_pred.astype(np.uint8)
    total = y_true.sum() + y_pred.sum()
    if total == 0:
        return 1.0
    intersection = np.sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (total + smooth)

def iou_numpy(y_true, y_pred, smooth=1e-6):
    y_true = y_true.astype(np.uint8)
    y_pred = y_pred.astype(np.uint8)
    union = np.sum((y_true + y_pred) > 0)
    if union == 0:
        return 1.0
    intersection = np.sum(y_true * y_pred)
    return (intersection + smooth) / (union + smooth)

def normalize_map(array_2d):
    arr = np.asarray(array_2d, dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr_min = float(arr.min())
    arr_max = float(arr.max())
    if arr_max - arr_min < 1e-8:
        return np.zeros_like(arr, dtype=np.float32)
    return (arr - arr_min) / (arr_max - arr_min)

def infer_model_outputs_logits(model, configured_value=None):
    """Return True only when the model output should be converted with sigmoid.

    The current project models usually end with a sigmoid activation, so their
    outputs are already probabilities. This guard prevents accidentally applying
    sigmoid twice, while still supporting future models that output logits.
    """
    if configured_value is not None:
        return bool(configured_value)
    activation = getattr(model.layers[-1], "activation", None)
    activation_name = getattr(activation, "__name__", "")
    return activation_name in {"linear", "identity"}

def get_resolved_isq_outputs_logits(outputs_logits=None):
    """Return the resolved logits/probability flag used by score functions."""
    resolved_value = ISQ_MODEL_OUTPUTS_LOGITS if outputs_logits is None else outputs_logits
    if resolved_value is None:
        raise RuntimeError(
            "ISQ_MODEL_OUTPUTS_LOGITS has not been resolved yet. "
            "Load the ischemia model with load_segmentation_models() before scoring."
        )
    return bool(resolved_value)

def output_to_probability_tf(model_output_tf, outputs_logits=None):
    values = tf.cast(model_output_tf, tf.float32)
    if get_resolved_isq_outputs_logits(outputs_logits):
        values = tf.math.sigmoid(values)
    return tf.clip_by_value(values, 0.0, 1.0)

def output_to_probability_np(model_output_np, outputs_logits=None):
    values = np.asarray(model_output_np, dtype=np.float32)
    if get_resolved_isq_outputs_logits(outputs_logits):
        values = 1.0 / (1.0 + np.exp(-values))
    return np.clip(values, 0.0, 1.0).astype(np.float32)

def parse_slice_idx_from_path(path_obj):
    stem = Path(path_obj).stem
    prefix = stem.split("-")[0]
    if prefix.isdigit():
        return int(prefix) - 1
    numbers = re.findall(r"\d+", stem)
    if not numbers:
        raise ValueError(f"Could not parse slice index from: {path_obj}")
    return int(numbers[0]) - 1

def load_binary_mask(mask_path):
    image = Image.open(mask_path).convert("L")
    mask = np.asarray(image, dtype=np.uint8)
    return (mask > 127).astype(np.uint8)

def load_union_mask_dict(mask_dir, vol_shape=None):
    mask_dir = Path(mask_dir)
    union_masks = {}
    if not mask_dir.exists():
        return union_masks
    for png_path in sorted(mask_dir.glob("*.png")):
        slice_idx = parse_slice_idx_from_path(png_path)
        if vol_shape is not None and (slice_idx < 0 or slice_idx >= vol_shape[2]):
            continue
        current_mask = load_binary_mask(png_path)
        if slice_idx in union_masks:
            union_masks[slice_idx] = np.maximum(union_masks[slice_idx], current_mask)
        else:
            union_masks[slice_idx] = current_mask
    return union_masks

def load_nifti_volume(nifti_path):
    nii = nib.load(str(nifti_path))
    volume = nii.get_fdata()
    if volume.ndim == 4:
        volume = np.squeeze(volume)
    if volume.ndim != 3:
        raise ValueError(f"Expected a 3D volume after squeeze, got shape {volume.shape} for {nifti_path}")
    volume = volume.astype(np.float32)
    volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)
    volume = np.rot90(volume, k=1, axes=(0, 1))
    volume = np.flip(volume, axis=0)
    return nii, volume

def resize_image_slice(slice_img, target_size=IMG_TARGET):
    slice_img = np.asarray(slice_img, dtype=np.float32)
    if slice_img.shape != (target_size, target_size):
        image = Image.fromarray(np.clip(slice_img * 255.0, 0, 255).astype(np.uint8))
        image = image.resize((target_size, target_size))
        slice_img = np.asarray(image, dtype=np.float32) / 255.0
    return slice_img.astype(np.float32)

def resize_mask(mask, target_size=IMG_TARGET):
    mask = np.asarray(mask, dtype=np.uint8)
    if mask.shape != (target_size, target_size):
        image = Image.fromarray(mask)
        image = image.resize((target_size, target_size), resample=Image.NEAREST)
        mask = np.asarray(image, dtype=np.uint8)
    return (mask > 0).astype(np.uint8)

def build_target_roi(prob_map, threshold=ISQ_THRESHOLD, allow_visual_fallback=ALLOW_TOPK_FALLBACK_FOR_VISUALIZATION, top_fraction=TOPK_FALLBACK_FRACTION, min_pixels=TOPK_FALLBACK_MIN_PIXELS):
    """Build the target ROI from the model prediction, not from ground truth.

    For quantitative XAI we explain the decision actually made by the model: the
    predicted lesion mask. If the prediction is empty there is no foreground
    decision to explain, so faithfulness metrics must be skipped instead of using
    a top-probability fallback that would invent a target.
    """
    prob_map = np.asarray(prob_map, dtype=np.float32)
    pred_mask = (prob_map > threshold).astype(np.uint8)
    if pred_mask.sum() > 0:
        return pred_mask, "predicted_mask", True

    if not allow_visual_fallback:
        return np.zeros_like(pred_mask, dtype=np.uint8), "no_predicted_lesion", False

    total_pixels = prob_map.size
    top_k = max(int(math.ceil(total_pixels * float(top_fraction))), int(min_pixels))
    top_k = min(top_k, total_pixels)

    flat = prob_map.reshape(-1)
    ranked_idx = np.argsort(-flat)
    roi_flat = np.zeros_like(flat, dtype=np.uint8)
    roi_flat[ranked_idx[:top_k]] = 1
    return roi_flat.reshape(prob_map.shape), "top_probability_fallback_visual_only", False

def segmentation_score_from_model_output(model_output_tf, roi_mask_tf, eps=1e-8):
    """Mean foreground probability inside the predicted lesion mask.

    This scalar is stable across different lesion sizes because it averages
    probability inside the fixed ROI instead of summing over pixels.
    """
    prob_map_tf = output_to_probability_tf(model_output_tf)
    roi_mask_tf = tf.cast(roi_mask_tf, tf.float32)
    denominator = tf.reduce_sum(roi_mask_tf) + eps
    return tf.reduce_sum(prob_map_tf * roi_mask_tf) / denominator

def compute_scalar_score_numpy(model_output_np, roi_mask_np, eps=1e-8):
    roi = np.asarray(roi_mask_np, dtype=np.float32)
    if float(roi.sum()) <= 0.0:
        return np.nan
    prob = output_to_probability_np(model_output_np)
    return float(np.sum(prob * roi) / (np.sum(roi) + eps))

def make_blurred_baseline(image_2d, radius=3.0):
    image_uint8 = np.clip(np.asarray(image_2d, dtype=np.float32) * 255.0, 0, 255).astype(np.uint8)
    blurred = Image.fromarray(image_uint8).filter(ImageFilter.GaussianBlur(radius=radius))
    return np.asarray(blurred, dtype=np.float32) / 255.0

def make_perturbation_baseline(image_2d, mode=PERTURBATION_BASELINE):
    """Baseline used by Deletion/Insertion and occlusion perturbations.

    Both "black" and "mean" are supported. The default "mean" usually gives
    smoother MRI perturbation curves because it avoids adding hard black squares
    that can behave like artificial edges.
    """
    image = np.asarray(image_2d, dtype=np.float32)
    if mode == "black":
        return np.zeros_like(image, dtype=np.float32)
    if mode == "mean":
        mean_value = float(np.mean(image))
        return np.full_like(image, mean_value, dtype=np.float32)
    if mode == "blurred":
        return make_blurred_baseline(image)
    raise ValueError(f"Unsupported PERTURBATION_BASELINE={mode!r}. Use 'mean', 'black', or 'blurred'.")

def build_sliding_positions(size, patch, stride):
    positions = list(range(0, max(size - patch + 1, 1), stride))
    last_start = max(size - patch, 0)
    if positions[-1] != last_start:
        positions.append(last_start)
    return positions

def get_edge_position_label(slice_idx, total_slices):
    if slice_idx < min(3, total_slices):
        return f"first_{slice_idx + 1}"
    if slice_idx >= max(total_slices - 3, 0):
        return f"last_{total_slices - slice_idx}"
    return None

def sort_case_dirs(dataset_dir):
    dataset_path = Path(dataset_dir)
    case_dirs = [p for p in dataset_path.iterdir() if p.is_dir() and p.name.startswith("CASE_")]
    def case_sort_key(path_obj):
        parts = path_obj.name.split("_")
        if len(parts) > 1 and parts[1].isdigit():
            return int(parts[1])
        return path_obj.name
    return sorted(case_dirs, key=case_sort_key)

def resolve_target_layer_name(model, requested_name=None):
    if requested_name is not None:
        _ = model.get_layer(requested_name)
        return requested_name

    output_conv_idx = None
    for idx in range(len(model.layers) - 1, -1, -1):
        layer = model.layers[idx]
        if isinstance(layer, Conv2D) and getattr(layer, "filters", None) == 1 and tuple(layer.kernel_size) == (1, 1):
            output_conv_idx = idx
            break

    search_start = output_conv_idx - 1 if output_conv_idx is not None else len(model.layers) - 1
    for idx in range(search_start, -1, -1):
        layer = model.layers[idx]
        if isinstance(layer, Conv2D) and getattr(layer, "filters", 0) > 1:
            return layer.name

    raise ValueError("Could not resolve a decoder convolution layer before the final 1x1 output convolution.")

def load_segmentation_models(model_isq_path, model_brain_path, use_brain_mask=True, target_layer_name=None):
    global ISQ_MODEL_OUTPUTS_LOGITS
    custom_objects = {
        "dice_coef": dice_coef,
        "dice_loss": dice_loss,
        "bce_dice_loss": bce_dice_loss,
    }
    model_isq = load_model(model_isq_path, custom_objects=custom_objects)
    model_brain = None
    if use_brain_mask:
        model_brain = load_model(model_brain_path, custom_objects=custom_objects)
    resolved_target_layer = resolve_target_layer_name(model_isq, requested_name=target_layer_name)
    detected_isq_outputs_logits = infer_model_outputs_logits(model_isq, ISQ_MODEL_OUTPUTS_LOGITS)
    ISQ_MODEL_OUTPUTS_LOGITS = bool(detected_isq_outputs_logits)
    return model_brain, model_isq, resolved_target_layer, ISQ_MODEL_OUTPUTS_LOGITS

def score_batch_inputs(model, batch_inputs, roi_mask, batch_size=SCORING_BATCH_SIZE):
    roi_tf = tf.convert_to_tensor(roi_mask, dtype=tf.float32)
    if float(np.sum(roi_mask)) <= 0.0:
        return np.full((len(batch_inputs),), np.nan, dtype=np.float32)

    score_chunks = []
    total = len(batch_inputs)

    for start in range(0, total, max(int(batch_size), 1)):
        stop = min(start + max(int(batch_size), 1), total)
        inputs_tf = tf.convert_to_tensor(batch_inputs[start:stop], dtype=tf.float32)
        predictions = model(inputs_tf, training=False)[..., 0]
        scores = []
        for batch_idx in range(predictions.shape[0]):
            scores.append(segmentation_score_from_model_output(predictions[batch_idx], roi_tf))
        scores = tf.stack(scores)
        score_chunks.append(scores.numpy().astype(np.float32))
        del inputs_tf
        del predictions
        del scores

    if not score_chunks:
        return np.asarray([], dtype=np.float32)
    return np.concatenate(score_chunks, axis=0)

def predict_ischemia_slice(slice_img, model_brain, model_isq, use_brain_mask=True):
    resized_img = resize_image_slice(slice_img, target_size=IMG_TARGET)
    input_tensor = resized_img[np.newaxis, :, :, np.newaxis].astype(np.float32)

    if use_brain_mask:
        if model_brain is None:
            raise ValueError("USE_BRAIN_MASK=True but no brain model was loaded.")
        brain_prob_map = model_brain.predict(input_tensor, verbose=0)[0, :, :, 0].astype(np.float32)
        brain_pred_mask = (brain_prob_map > BRAIN_THRESHOLD).astype(np.uint8)
        isq_input_tensor = input_tensor * brain_pred_mask[np.newaxis, :, :, np.newaxis].astype(np.float32)
    else:
        brain_prob_map = np.ones((IMG_TARGET, IMG_TARGET), dtype=np.float32)
        brain_pred_mask = np.ones((IMG_TARGET, IMG_TARGET), dtype=np.uint8)
        isq_input_tensor = input_tensor.copy()

    isq_raw_output = model_isq.predict(isq_input_tensor, verbose=0)[0, :, :, 0].astype(np.float32)
    isq_prob_map = output_to_probability_np(isq_raw_output)
    isq_pred_mask = (isq_prob_map > ISQ_THRESHOLD).astype(np.uint8)
    roi_mask, roi_strategy, has_explainable_target = build_target_roi(isq_prob_map)
    baseline_image = make_perturbation_baseline(isq_input_tensor[0, :, :, 0])
    base_score = compute_scalar_score_numpy(isq_raw_output, roi_mask)

    return {
        "resized_img": resized_img,
        "input_tensor": input_tensor,
        "brain_prob_map": brain_prob_map,
        "brain_pred_mask": brain_pred_mask,
        "isq_input_tensor": isq_input_tensor,
        "isq_raw_output": isq_raw_output,
        "isq_prob_map": isq_prob_map,
        "isq_pred_mask": isq_pred_mask,
        "roi_mask": roi_mask.astype(np.uint8),
        "roi_strategy": roi_strategy,
        "has_explainable_target": bool(has_explainable_target),
        "baseline_image": baseline_image,
        "base_score": base_score,
    }


## 5. Métodos XAI y métricas de fidelidad

Aquí se calculan los mapas de atribución y las pruebas de perturbación. Como los métodos CAM e Integrated Gradients no producen por sí solos una métrica de calidad clínica, se complementan con curvas de Deletion, Insertion, MoRF, LeRF y AOPC.

Estas métricas indican cuánto cambia la puntuación del modelo al retirar o introducir regiones según la importancia asignada por cada método. Su interpretación depende del protocolo de perturbación y del objetivo escalar definido para la segmentación.


In [ ]:
# ============================================================
# Helper functions: XAI maps and faithfulness curves
# ============================================================

def compute_gradcam(model, input_tensor, roi_mask, target_layer_name):
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(target_layer_name).output, model.output],
    )
    input_tf = tf.convert_to_tensor(input_tensor, dtype=tf.float32)
    roi_tf = tf.convert_to_tensor(roi_mask, dtype=tf.float32)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(input_tf, training=False)
        score = segmentation_score_from_model_output(predictions[0, :, :, 0], roi_tf)

    grads = tape.gradient(score, conv_outputs)
    if grads is None:
        raise RuntimeError("Grad-CAM gradients are None. Check the target layer and score definition.")

    conv_outputs = conv_outputs[0]
    grads = grads[0]
    weights = tf.reduce_mean(grads, axis=(0, 1))
    cam = tf.reduce_sum(conv_outputs * weights, axis=-1)
    cam = tf.nn.relu(cam)
    cam = tf.image.resize(cam[..., tf.newaxis], (IMG_TARGET, IMG_TARGET), method="bilinear")[..., 0]
    return normalize_map(cam.numpy())

def compute_gradcam_pp(model, input_tensor, roi_mask, target_layer_name):
    """
    Memory-safe Grad-CAM++ approximation.

    Exact higher-order Grad-CAM++ can be extremely memory hungry in TensorFlow for
    segmentation models. This implementation keeps the Grad-CAM++ weighting scheme
    but approximates the alpha terms from positive first-order gradients to avoid
    retaining second- and third-order graphs in RAM.
    """
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(target_layer_name).output, model.output],
    )
    input_tf = tf.convert_to_tensor(input_tensor, dtype=tf.float32)
    roi_tf = tf.convert_to_tensor(roi_mask, dtype=tf.float32)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(input_tf, training=False)
        score = segmentation_score_from_model_output(predictions[0, :, :, 0], roi_tf)

    first_grads = tape.gradient(score, conv_outputs)
    if first_grads is None:
        raise RuntimeError("Grad-CAM++ gradients are None.")

    conv_outputs = conv_outputs[0]
    first_grads = tf.nn.relu(first_grads[0])
    grads_sq = tf.square(first_grads)
    grads_cube = grads_sq * first_grads

    global_sum = tf.reduce_sum(conv_outputs, axis=(0, 1), keepdims=True)
    alpha_denom = 2.0 * grads_sq + global_sum * grads_cube
    alpha_denom = tf.where(alpha_denom > 1e-8, alpha_denom, tf.ones_like(alpha_denom))
    alphas = grads_sq / alpha_denom
    weights = tf.reduce_sum(alphas * first_grads, axis=(0, 1))

    cam = tf.reduce_sum(weights * conv_outputs, axis=-1)
    cam = tf.nn.relu(cam)
    cam = tf.image.resize(cam[..., tf.newaxis], (IMG_TARGET, IMG_TARGET), method="bilinear")[..., 0]
    return normalize_map(cam.numpy())

def compute_integrated_gradients(model, input_tensor, roi_mask, steps=IG_STEPS, baseline_image=None):
    """Memory-safe Integrated Gradients computed incrementally."""
    input_single = tf.convert_to_tensor(input_tensor[0], dtype=tf.float32)
    if baseline_image is None:
        baseline_single = tf.zeros_like(input_single)
    else:
        baseline = np.asarray(baseline_image, dtype=np.float32)[..., np.newaxis]
        baseline_single = tf.convert_to_tensor(baseline, dtype=tf.float32)
    roi_tf = tf.convert_to_tensor(roi_mask, dtype=tf.float32)

    alphas = np.linspace(0.0, 1.0, int(steps) + 1, dtype=np.float32)
    grad_steps = []

    for alpha in alphas:
        interpolated = baseline_single + alpha * (input_single - baseline_single)
        with tf.GradientTape() as tape:
            tape.watch(interpolated)
            prediction = model(interpolated[None, ...], training=False)[0, :, :, 0]
            score = segmentation_score_from_model_output(prediction, roi_tf)
        grad = tape.gradient(score, interpolated)
        grad_steps.append(grad.numpy().astype(np.float32)[..., 0])
        del interpolated
        del prediction
        del score
        del grad

    grads = np.stack(grad_steps, axis=0)
    avg_grads = 0.5 * (grads[:-1] + grads[1:])
    mean_grads = np.mean(avg_grads, axis=0)
    integrated = (input_single.numpy()[..., 0] - baseline_single.numpy()[..., 0]) * mean_grads
    attribution = np.maximum(integrated, 0.0)
    return normalize_map(attribution)

def compute_occlusion_map(model, input_tensor, roi_mask, patch=OCCLUSION_PATCH, stride=OCCLUSION_STRIDE, baseline_image=None):
    original_image = input_tensor[0, :, :, 0].astype(np.float32)
    if float(np.sum(roi_mask)) <= 0.0:
        summary = {
            "occlusion_base_score": np.nan,
            "occlusion_mean_drop": np.nan,
            "occlusion_max_drop": np.nan,
            "occlusion_sum_drop": np.nan,
        }
        return np.zeros_like(original_image, dtype=np.float32), summary

    baseline_image = make_perturbation_baseline(original_image) if baseline_image is None else baseline_image
    base_score = compute_scalar_score_numpy(model.predict(input_tensor, verbose=0)[0, :, :, 0], roi_mask)

    height, width = original_image.shape
    y_positions = build_sliding_positions(height, patch, stride)
    x_positions = build_sliding_positions(width, patch, stride)

    perturbed_images = []
    coords = []
    for y0 in y_positions:
        y1 = min(y0 + patch, height)
        for x0 in x_positions:
            x1 = min(x0 + patch, width)
            perturbed = original_image.copy()
            perturbed[y0:y1, x0:x1] = baseline_image[y0:y1, x0:x1]
            perturbed_images.append(perturbed)
            coords.append((y0, y1, x0, x1))

    perturbed_batch = np.stack(perturbed_images, axis=0)[..., np.newaxis].astype(np.float32)
    perturbed_scores = score_batch_inputs(model, perturbed_batch, roi_mask)

    occlusion_values = np.zeros_like(original_image, dtype=np.float32)
    occlusion_counts = np.zeros_like(original_image, dtype=np.float32)
    for score_value, (y0, y1, x0, x1) in zip(perturbed_scores, coords):
        drop = float(base_score - score_value)
        occlusion_values[y0:y1, x0:x1] += drop
        occlusion_counts[y0:y1, x0:x1] += 1.0

    occlusion_map = occlusion_values / np.maximum(occlusion_counts, 1.0)
    occlusion_positive = np.maximum(occlusion_map, 0.0)
    summary = {
        "occlusion_base_score": float(base_score),
        "occlusion_mean_drop": float(np.mean(occlusion_map)),
        "occlusion_max_drop": float(np.max(occlusion_map)),
        "occlusion_sum_drop": float(np.sum(occlusion_positive)),
    }
    return normalize_map(occlusion_positive), summary

def create_curve_batch(original_image, baseline_image, sorted_indices, curve_type, curve_steps=CURVE_STEPS):
    flat_original = original_image.reshape(-1)
    flat_baseline = baseline_image.reshape(-1)
    num_pixels = flat_original.size
    counts = np.round(np.linspace(0, num_pixels, curve_steps + 1)).astype(int)
    fractions = np.linspace(0.0, 1.0, curve_steps + 1, dtype=np.float32)

    batch_images = []
    for count in counts:
        if curve_type in ("deletion", "morf", "lerf"):
            working = flat_original.copy()
            working[sorted_indices[:count]] = flat_baseline[sorted_indices[:count]]
        elif curve_type == "insertion":
            working = flat_baseline.copy()
            working[sorted_indices[:count]] = flat_original[sorted_indices[:count]]
        else:
            raise ValueError(f"Unsupported curve_type: {curve_type}")
        batch_images.append(working.reshape(original_image.shape))

    batch_inputs = np.stack(batch_images, axis=0)[..., np.newaxis].astype(np.float32)
    return fractions, batch_inputs

def trapezoid_auc(y_values, x_values):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y_values, x_values)
    return np.trapz(y_values, x_values)

def empty_curve(curve_steps=CURVE_STEPS):
    fractions = np.linspace(0.0, 1.0, int(curve_steps) + 1, dtype=np.float32)
    scores = np.full_like(fractions, np.nan, dtype=np.float32)
    return {"fractions": fractions, "scores": scores, "raw_scores": scores.copy(), "auc": np.nan}

def compute_curve_from_attribution(model, input_tensor, roi_mask, attribution_map, curve_type, baseline_image=None, curve_steps=CURVE_STEPS):
    original_image = input_tensor[0, :, :, 0].astype(np.float32)
    baseline_image = make_perturbation_baseline(original_image) if baseline_image is None else baseline_image
    attribution_flat = np.asarray(attribution_map, dtype=np.float32).reshape(-1)

    descending = curve_type in ("deletion", "insertion", "morf")
    sorted_indices = np.argsort(-attribution_flat) if descending else np.argsort(attribution_flat)
    fractions, batch_inputs = create_curve_batch(original_image, baseline_image, sorted_indices, curve_type, curve_steps=curve_steps)
    raw_scores = score_batch_inputs(model, batch_inputs, roi_mask)

    original_score = raw_scores[0] if curve_type in ("deletion", "morf", "lerf") else raw_scores[-1]
    baseline_score = raw_scores[0] if curve_type == "insertion" else np.nan

    if curve_type == "insertion":
        denominator = float(original_score) - float(baseline_score)
        if not np.isfinite(denominator) or abs(denominator) < 1e-8:
            normalized_scores = np.full_like(raw_scores, np.nan, dtype=np.float32)
            auc_value = np.nan
        else:
            normalized_scores = (raw_scores - float(baseline_score)) / (denominator + 1e-8)
            auc_value = float(trapezoid_auc(normalized_scores, fractions))
    else:
        if not np.isfinite(original_score) or abs(float(original_score)) < 1e-8:
            normalized_scores = np.full_like(raw_scores, np.nan, dtype=np.float32)
            auc_value = np.nan
        else:
            normalized_scores = raw_scores / (float(original_score) + 1e-8)
            auc_value = float(trapezoid_auc(normalized_scores, fractions))

    return {
        "fractions": fractions.astype(np.float32),
        "scores": normalized_scores.astype(np.float32),
        "raw_scores": raw_scores.astype(np.float32),
        "original_score": float(original_score) if np.isfinite(original_score) else np.nan,
        "baseline_score": float(baseline_score) if np.isfinite(baseline_score) else np.nan,
        "auc": auc_value,
    }

def assert_faithfulness_curve_sanity(faithfulness_bundle, context="faithfulness", endpoint_tol=5e-3, auc_tol=1e-6):
    """Validate normalized curves used for quantitative XAI metrics."""
    deletion = np.asarray(faithfulness_bundle["deletion"]["scores"], dtype=np.float32)
    insertion = np.asarray(faithfulness_bundle["insertion"]["scores"], dtype=np.float32)

    if np.isfinite(deletion).any() and not np.isclose(deletion[0], 1.0, atol=endpoint_tol):
        raise AssertionError(f"{context}: Deletion curve should start near 1, got {deletion[0]:.6f}")
    if np.isfinite(insertion).any():
        if not np.isclose(insertion[0], 0.0, atol=endpoint_tol):
            raise AssertionError(f"{context}: Insertion curve should start near 0, got {insertion[0]:.6f}")
        if not np.isclose(insertion[-1], 1.0, atol=endpoint_tol):
            raise AssertionError(f"{context}: Insertion curve should end near 1, got {insertion[-1]:.6f}")

    for curve_name in ("deletion", "insertion", "morf", "lerf"):
        auc_value = faithfulness_bundle[curve_name]["auc"]
        if np.isfinite(auc_value) and auc_value < -auc_tol:
            raise AssertionError(f"{context}: {curve_name} AUC should be non-negative, got {auc_value:.6f}")

def compute_faithfulness_bundle(model, input_tensor, roi_mask, attribution_map, baseline_image=None, curve_steps=CURVE_STEPS):
    if float(np.sum(roi_mask)) <= 0.0:
        return {
            "deletion": empty_curve(curve_steps),
            "insertion": empty_curve(curve_steps),
            "morf": empty_curve(curve_steps),
            "lerf": empty_curve(curve_steps),
            "aopc": np.nan,
            "normalized_aopc": np.nan,
        }

    baseline_image = make_perturbation_baseline(input_tensor[0, :, :, 0]) if baseline_image is None else baseline_image
    deletion = compute_curve_from_attribution(model, input_tensor, roi_mask, attribution_map, "deletion", baseline_image=baseline_image, curve_steps=curve_steps)
    insertion = compute_curve_from_attribution(model, input_tensor, roi_mask, attribution_map, "insertion", baseline_image=baseline_image, curve_steps=curve_steps)
    morf = compute_curve_from_attribution(model, input_tensor, roi_mask, attribution_map, "morf", baseline_image=baseline_image, curve_steps=curve_steps)
    lerf = compute_curve_from_attribution(model, input_tensor, roi_mask, attribution_map, "lerf", baseline_image=baseline_image, curve_steps=curve_steps)

    base_score = float(morf["raw_scores"][0])
    raw_drops = base_score - morf["raw_scores"][1:]
    aopc = float(np.nanmean(raw_drops)) if len(raw_drops) > 0 else np.nan
    normalized_aopc = float(np.nanmean(1.0 - morf["scores"][1:])) if len(morf["scores"]) > 1 else np.nan

    faithfulness_bundle = {
        "deletion": deletion,
        "insertion": insertion,
        "morf": morf,
        "lerf": lerf,
        "aopc": aopc,
        "normalized_aopc": normalized_aopc,
    }
    assert_faithfulness_curve_sanity(faithfulness_bundle)
    return faithfulness_bundle

def compute_xai_bundle(
    model,
    input_tensor,
    roi_mask,
    target_layer_name,
    baseline_image=None,
    ig_steps=IG_STEPS,
    curve_steps=CURVE_STEPS,
    occlusion_patch=OCCLUSION_PATCH,
    occlusion_stride=OCCLUSION_STRIDE,
    include_occlusion=True,
    faithfulness_methods=None,
):
    baseline_image = make_perturbation_baseline(input_tensor[0, :, :, 0]) if baseline_image is None else baseline_image

    if float(np.sum(roi_mask)) <= 0.0:
        attribution_maps = {
            "gradcam": np.zeros((IMG_TARGET, IMG_TARGET), dtype=np.float32),
            "gradcampp": np.zeros((IMG_TARGET, IMG_TARGET), dtype=np.float32),
            "ig": np.zeros((IMG_TARGET, IMG_TARGET), dtype=np.float32),
        }
    else:
        attribution_maps = {
            "gradcam": compute_gradcam(model, input_tensor, roi_mask, target_layer_name),
            "gradcampp": compute_gradcam_pp(model, input_tensor, roi_mask, target_layer_name),
            "ig": compute_integrated_gradients(model, input_tensor, roi_mask, steps=ig_steps, baseline_image=baseline_image),
        }

    if include_occlusion:
        occlusion_map, occlusion_summary = compute_occlusion_map(
            model,
            input_tensor,
            roi_mask,
            patch=occlusion_patch,
            stride=occlusion_stride,
            baseline_image=baseline_image,
        )
    else:
        occlusion_map = np.zeros((IMG_TARGET, IMG_TARGET), dtype=np.float32)
        occlusion_summary = {
            "occlusion_base_score": np.nan,
            "occlusion_mean_drop": np.nan,
            "occlusion_max_drop": np.nan,
            "occlusion_sum_drop": np.nan,
        }

    faithfulness_methods = tuple(attribution_maps.keys()) if faithfulness_methods is None else tuple(faithfulness_methods)
    faithfulness_by_method = {}
    for method_name, attribution_map in attribution_maps.items():
        if method_name not in faithfulness_methods:
            continue
        faithfulness_by_method[method_name] = compute_faithfulness_bundle(
            model,
            input_tensor,
            roi_mask,
            attribution_map,
            baseline_image=baseline_image,
            curve_steps=curve_steps,
        )
        assert_faithfulness_curve_sanity(faithfulness_by_method[method_name], context=f"{method_name}")

    return {
        "attribution_maps": attribution_maps,
        "occlusion_map": occlusion_map,
        "occlusion_summary": occlusion_summary,
        "faithfulness_by_method": faithfulness_by_method,
    }

def assert_normalized_map(name, array_2d):
    arr = np.asarray(array_2d, dtype=np.float32)
    if arr.shape != (IMG_TARGET, IMG_TARGET):
        raise AssertionError(f"{name} must have shape {(IMG_TARGET, IMG_TARGET)}, got {arr.shape}")
    if not np.isfinite(arr).all():
        raise AssertionError(f"{name} contains non-finite values")
    if float(arr.min()) < -1e-6 or float(arr.max()) > 1.0 + 1e-6:
        raise AssertionError(f"{name} is not normalized to [0, 1]")


## 6. Selección de ejemplos, visualización y tablas

Estas funciones seleccionan cortes con lesión, cortes sin lesión y cortes extremos del volumen. La selección combina criterios cuantitativos y casos potencialmente problemáticos, como falsos positivos o cortes con alta carga predicha.

Las figuras resultantes permiten revisar de forma conjunta la imagen, la máscara manual, la predicción y las explicaciones, evitando basar la discusión solo en promedios numéricos.


In [ ]:
# ============================================================
# Helper functions: selection, plotting, and export tables
# ============================================================

def collect_smoke_examples(dataset_dir):
    lesion_example = None
    non_lesion_example = None

    for case_dir in sort_case_dirs(dataset_dir):
        nifti_path = case_dir / "Seq No.nii"
        if not nifti_path.exists():
            continue
        _, volume = load_nifti_volume(nifti_path)
        lesion_masks = load_union_mask_dict(case_dir / "isquemia", volume.shape)

        for slice_idx in range(volume.shape[2]):
            gt_mask = resize_mask(lesion_masks.get(slice_idx, np.zeros_like(volume[:, :, slice_idx], dtype=np.uint8)))
            example = {
                "case": case_dir.name,
                "slice_idx": slice_idx,
                "slice_img": volume[:, :, slice_idx],
                "gt_mask": gt_mask,
            }
            if gt_mask.sum() > 0 and lesion_example is None:
                lesion_example = example
            if gt_mask.sum() == 0 and non_lesion_example is None:
                non_lesion_example = example
            if lesion_example is not None and non_lesion_example is not None:
                return lesion_example, non_lesion_example

    return lesion_example, non_lesion_example

def pick_quantile_rows(df, column, quantiles):
    if df.empty:
        return []
    ordered = df.sort_values(column).reset_index(drop=False)
    chosen = []
    for quantile in quantiles:
        target_value = ordered[column].quantile(quantile)
        idx = (ordered[column] - target_value).abs().idxmin()
        original_idx = int(ordered.loc[idx, "index"])
        if original_idx not in chosen:
            chosen.append(original_idx)
    return chosen

def select_lesion_slices(slice_summary_df, limit=SELECTED_SLICES_PER_GROUP):
    subset = slice_summary_df[slice_summary_df["is_lesion_gt"]].copy()
    if subset.empty:
        return []

    selected_indices = pick_quantile_rows(subset, "isq_dice", [0.10, 0.30, 0.50, 0.70, 0.90])
    largest_idx = int(subset.sort_values("gt_lesion_pixels", ascending=False).index[0])
    if largest_idx not in selected_indices:
        selected_indices.append(largest_idx)

    backfill = subset.sort_values(["gt_lesion_pixels", "pred_lesion_pixels"], ascending=[False, False]).index.tolist()
    for idx in backfill:
        if idx not in selected_indices:
            selected_indices.append(int(idx))
        if len(selected_indices) >= limit:
            break

    selected_indices = selected_indices[:limit]
    records = []
    for idx in selected_indices:
        row = slice_summary_df.loc[idx]
        reason = "dice_quantile" if idx != largest_idx else "largest_gt_lesion"
        records.append({
            "slice_uid": row["slice_uid"],
            "case": row["case"],
            "slice_idx_original": int(row["slice_idx_original"]),
            "selection_group": "lesion",
            "selection_reason": reason,
        })
    return records

def select_nonlesion_slices(slice_summary_df, limit=SELECTED_SLICES_PER_GROUP):
    subset = slice_summary_df[slice_summary_df["is_non_lesion_gt"]].copy()
    if subset.empty:
        return []

    records = []
    chosen = []

    top_pred = subset.sort_values(["pred_lesion_pixels", "base_score"], ascending=[False, False]).head(min(3, limit))
    for idx, row in top_pred.iterrows():
        chosen.append(int(idx))
        records.append({
            "slice_uid": row["slice_uid"],
            "case": row["case"],
            "slice_idx_original": int(row["slice_idx_original"]),
            "selection_group": "non_lesion",
            "selection_reason": "highest_predicted_burden",
        })

    remaining = subset.drop(index=chosen, errors="ignore")
    remaining_slots = max(limit - len(records), 0)
    if remaining_slots > 0 and not remaining.empty:
        residual_median = remaining["base_score"].median()
        median_rows = remaining.assign(median_distance=(remaining["base_score"] - residual_median).abs())
        median_rows = median_rows.sort_values(["median_distance", "pred_lesion_pixels"], ascending=[True, False]).head(remaining_slots)
        for idx, row in median_rows.iterrows():
            records.append({
                "slice_uid": row["slice_uid"],
                "case": row["case"],
                "slice_idx_original": int(row["slice_idx_original"]),
                "selection_group": "non_lesion",
                "selection_reason": "median_residual_score",
            })
            chosen.append(int(idx))

    if len(records) < limit:
        backfill = subset.drop(index=chosen, errors="ignore").sort_values(["pred_lesion_pixels", "base_score"], ascending=[False, False])
        for _, row in backfill.iterrows():
            records.append({
                "slice_uid": row["slice_uid"],
                "case": row["case"],
                "slice_idx_original": int(row["slice_idx_original"]),
                "selection_group": "non_lesion",
                "selection_reason": "backfill",
            })
            if len(records) >= limit:
                break
    return records[:limit]

def select_edge_slices(slice_summary_df, limit=SELECTED_SLICES_PER_GROUP):
    subset = slice_summary_df[slice_summary_df["is_edge_first_last3"]].copy()
    if subset.empty:
        return []

    edge_order = ["first_1", "first_2", "first_3", "last_3", "last_2", "last_1"]
    records = []
    used_uids = set()

    for edge_label in edge_order:
        edge_subset = subset[subset["edge_position_label"] == edge_label]
        if edge_subset.empty:
            continue
        lesion_subset = edge_subset[edge_subset["is_lesion_gt"]]
        candidate_pool = lesion_subset if not lesion_subset.empty else edge_subset
        candidate = candidate_pool.sort_values(["pred_lesion_pixels", "gt_lesion_pixels"], ascending=[False, False]).iloc[0]
        if candidate["slice_uid"] in used_uids:
            continue
        used_uids.add(candidate["slice_uid"])
        records.append({
            "slice_uid": candidate["slice_uid"],
            "case": candidate["case"],
            "slice_idx_original": int(candidate["slice_idx_original"]),
            "selection_group": "edge_first_last3",
            "selection_reason": f"edge_position_{edge_label}",
        })
        if len(records) >= limit:
            return records

    if len(records) < limit:
        backfill = subset[~subset["slice_uid"].isin(used_uids)].sort_values(["pred_lesion_pixels", "gt_lesion_pixels"], ascending=[False, False])
        for _, row in backfill.iterrows():
            records.append({
                "slice_uid": row["slice_uid"],
                "case": row["case"],
                "slice_idx_original": int(row["slice_idx_original"]),
                "selection_group": "edge_first_last3",
                "selection_reason": "backfill",
            })
            if len(records) >= limit:
                break
    return records[:limit]

def build_selected_slices(slice_summary_df):
    selected_records = []
    selected_records.extend(select_lesion_slices(slice_summary_df, limit=SELECTED_SLICES_PER_GROUP))
    selected_records.extend(select_nonlesion_slices(slice_summary_df, limit=SELECTED_SLICES_PER_GROUP))
    selected_records.extend(select_edge_slices(slice_summary_df, limit=SELECTED_SLICES_PER_GROUP))
    if not selected_records:
        return pd.DataFrame(columns=["slice_uid", "case", "slice_idx_original", "selection_group", "selection_reason"])
    return pd.DataFrame(selected_records)

def make_export_filename(case_name, slice_idx_original):
    safe_case = re.sub(r"[^A-Za-z0-9_\-]", "_", case_name)
    return f"{safe_case}_slice_{int(slice_idx_original):03d}.png"

def plot_heatmap_overlay(ax, image_2d, heatmap_2d, title, cmap="jet"):
    ax.imshow(image_2d, cmap="gray")
    ax.imshow(heatmap_2d, cmap=cmap, alpha=0.45, vmin=0.0, vmax=1.0)
    ax.set_title(title)
    ax.axis("off")

def render_slice_montage(artifact, output_path):
    image = artifact["image"]
    gt_mask = artifact["gt_mask"]
    gradcam = artifact["gradcam"]
    gradcampp = artifact["gradcampp"]
    ig_map = artifact["ig"]
    faithfulness = artifact["faithfulness_by_method"]
    meta = artifact["meta"]

    fig, axes = plt.subplots(1, 6, figsize=(24, 4.5))

    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(gt_mask, cmap="gray")
    axes[1].set_title("GT Mask")
    axes[1].axis("off")

    plot_heatmap_overlay(axes[2], image, gradcam, "Grad-CAM")
    plot_heatmap_overlay(axes[3], image, gradcampp, "Grad-CAM++")
    plot_heatmap_overlay(axes[4], image, ig_map, "Integrated Gradients")

    if meta.get("has_explainable_target", meta.get("pred_lesion_pixels", 0) > 0):
        for method_name, label in (("gradcam", "Grad-CAM"), ("gradcampp", "Grad-CAM++"), ("ig", "IG")):
            curve = faithfulness[method_name]["deletion"]
            axes[5].plot(
                curve["fractions"],
                curve["scores"],
                label=f"{label} (AUC={curve['auc']:.3f})",
                linewidth=2,
            )
        axes[5].set_title("Normalized Deletion Curve")
        axes[5].set_xlabel("Perturbed fraction")
        axes[5].set_ylabel("Score / original score")
        axes[5].set_ylim(bottom=0.0)
        axes[5].grid(alpha=0.25)
        axes[5].legend(fontsize=8)
    else:
        axes[5].axis("off")
        axes[5].text(0.5, 0.5, "No predicted lesion\nNo quantitative XAI target", ha="center", va="center")

    figure_title = (
        f"{meta['case']} | slice {meta['slice_idx_original']} | "
        f"GT px={meta['gt_lesion_pixels']} | Pred px={meta['pred_lesion_pixels']} | "
        f"target={meta['target_mask_strategy_used']}"
    )
    fig.suptitle(figure_title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

def get_curve_record_columns():
    return [
        "slice_uid",
        "case",
        "slice_idx_original",
        "method",
        "curve_type",
        "step_idx",
        "fraction",
        "score",
        "raw_score",
    ]

def append_curve_records(curve_records, output_path, write_header):
    if not curve_records:
        return write_header
    pd.DataFrame(curve_records, columns=get_curve_record_columns()).to_csv(
        output_path,
        mode="w" if write_header else "a",
        header=write_header,
        index=False,
    )
    return False

def build_export_artifact(selected_row, dataset_dir, model_brain, model_isq, target_layer_name, use_brain_mask=True):
    case_dir = Path(dataset_dir) / selected_row["case"]
    nifti_path = case_dir / "Seq No.nii"
    if not nifti_path.exists():
        raise FileNotFoundError(f"Missing NIfTI for export slice: {nifti_path}")

    _, volume = load_nifti_volume(nifti_path)
    slice_idx = int(selected_row["slice_idx_original"])
    if slice_idx < 0 or slice_idx >= volume.shape[2]:
        raise IndexError(f"Slice index {slice_idx} is out of range for {case_dir.name}")

    lesion_masks = load_union_mask_dict(case_dir / "isquemia", volume.shape)
    slice_img = volume[:, :, slice_idx]
    gt_mask_raw = lesion_masks.get(slice_idx, np.zeros_like(slice_img, dtype=np.uint8))
    gt_mask = resize_mask(gt_mask_raw, target_size=IMG_TARGET)

    prediction = predict_ischemia_slice(
        slice_img,
        model_brain=model_brain,
        model_isq=model_isq,
        use_brain_mask=use_brain_mask,
    )
    xai_bundle = compute_xai_bundle(
        model=model_isq,
        input_tensor=prediction["isq_input_tensor"],
        roi_mask=prediction["roi_mask"],
        target_layer_name=target_layer_name,
        baseline_image=prediction["baseline_image"],
        ig_steps=IG_STEPS,
        curve_steps=CURVE_STEPS,
    )

    artifact = {
        "image": prediction["resized_img"],
        "gt_mask": gt_mask,
        "gradcam": xai_bundle["attribution_maps"]["gradcam"],
        "gradcampp": xai_bundle["attribution_maps"]["gradcampp"],
        "ig": xai_bundle["attribution_maps"]["ig"],
        "occlusion": xai_bundle["occlusion_map"],
        "faithfulness_by_method": xai_bundle["faithfulness_by_method"],
        "meta": {
            "case": selected_row["case"],
            "slice_idx_original": slice_idx,
            "gt_lesion_pixels": int(gt_mask.sum()),
            "pred_lesion_pixels": int(prediction["isq_pred_mask"].sum()),
            "target_mask_strategy_used": prediction["roi_strategy"],
            "has_explainable_target": bool(prediction["has_explainable_target"]),
        },
    }

    del volume
    del lesion_masks
    del slice_img
    del gt_mask_raw
    del prediction
    del xai_bundle
    gc.collect()
    return artifact

def build_group_summary(method_metrics_df, slice_summary_df):
    numeric_columns = [
        "deletion_auc",
        "insertion_auc",
        "morf_auc",
        "lerf_auc",
        "aopc",
        "normalized_aopc",
        "occlusion_mean_drop",
        "occlusion_max_drop",
        "occlusion_sum_drop",
    ]
    merge_columns = [
        "slice_uid",
        "is_lesion_gt",
        "is_non_lesion_gt",
        "is_edge_first_last3",
        "has_explainable_target",
        "prediction_outcome",
    ]
    merged = method_metrics_df.merge(slice_summary_df[merge_columns], on="slice_uid", how="left")
    group_filters = {
        "lesion": merged["is_lesion_gt"].fillna(False),
        "non_lesion": merged["is_non_lesion_gt"].fillna(False),
        "edge_first_last3": merged["is_edge_first_last3"].fillna(False),
        "true_positive": merged["prediction_outcome"].eq("TP"),
        "false_positive": merged["prediction_outcome"].eq("FP"),
        "false_negative": merged["prediction_outcome"].eq("FN"),
        "true_negative": merged["prediction_outcome"].eq("TN"),
        "quantitative_xai_included": merged["has_explainable_target"].fillna(False),
    }
    rows = []
    for group_name, group_mask in group_filters.items():
        group_subset = merged[group_mask]
        if group_subset.empty:
            continue
        for method_name, method_subset in group_subset.groupby("method"):
            for metric_name in numeric_columns:
                metric_values = method_subset[metric_name].dropna()
                if metric_values.empty:
                    continue
                rows.append({
                    "group": group_name,
                    "method": method_name,
                    "metric": metric_name,
                    "n": int(metric_values.shape[0]),
                    "mean": float(metric_values.mean()),
                    "std": float(metric_values.std(ddof=0)),
                    "median": float(metric_values.median()),
                })
    return pd.DataFrame(rows)

def save_checkpoint_tables(slice_records, method_records, curve_records, curve_records_path=None):
    pd.DataFrame(slice_records).to_csv(CSV_DIR / "slice_summary_checkpoint.csv", index=False)
    pd.DataFrame(method_records).to_csv(CSV_DIR / "method_metrics_checkpoint.csv", index=False)
    curve_path = Path(curve_records_path) if curve_records_path is not None else CSV_DIR / "curve_points_checkpoint.csv"
    pd.DataFrame(curve_records, columns=get_curve_record_columns()).to_csv(curve_path, index=False)

def load_checkpoint_tables(slice_path=None, method_path=None, curve_path=None):
    slice_path = Path(slice_path) if slice_path is not None else CSV_DIR / "slice_summary_checkpoint.csv"
    method_path = Path(method_path) if method_path is not None else CSV_DIR / "method_metrics_checkpoint.csv"
    curve_path = Path(curve_path) if curve_path is not None else CSV_DIR / "curve_points_checkpoint.csv"

    if not slice_path.exists():
        raise FileNotFoundError(f"Missing slice checkpoint table: {slice_path}")
    if not method_path.exists():
        raise FileNotFoundError(f"Missing method checkpoint table: {method_path}")

    slice_summary_df = pd.read_csv(slice_path)
    method_metrics_df = pd.read_csv(method_path)
    if curve_path.exists():
        curve_points_df = pd.read_csv(curve_path)
    else:
        curve_points_df = pd.DataFrame(columns=get_curve_record_columns())

    # Backward-compatible loading: old checkpoints did not separate TP/FP/FN/TN
    # or raw/normalized curve scores. Prefer rerunning the heavy analysis after
    # this methodological update, but keep reload cells informative.
    if "has_explainable_target" not in slice_summary_df.columns and "pred_lesion_pixels" in slice_summary_df.columns:
        slice_summary_df["has_explainable_target"] = slice_summary_df["pred_lesion_pixels"].fillna(0).astype(int) > 0
    if "prediction_outcome" not in slice_summary_df.columns and {"gt_lesion_pixels", "pred_lesion_pixels"}.issubset(slice_summary_df.columns):
        gt_positive = slice_summary_df["gt_lesion_pixels"].fillna(0).astype(int) > 0
        pred_positive = slice_summary_df["pred_lesion_pixels"].fillna(0).astype(int) > 0
        slice_summary_df["prediction_outcome"] = np.select(
            [gt_positive & pred_positive, (~gt_positive) & pred_positive, gt_positive & (~pred_positive)],
            ["TP", "FP", "FN"],
            default="TN",
        )
    if "raw_score" not in curve_points_df.columns:
        curve_points_df["raw_score"] = curve_points_df["score"] if "score" in curve_points_df.columns else np.nan
    return slice_summary_df, method_metrics_df, curve_points_df


## 7. Carga de modelos y capa objetivo

Se cargan el modelo cerebral y el modelo de isquemia. También se determina la capa convolucional usada por Grad-CAM y Grad-CAM++, priorizando una capa adecuada para explicar la salida de segmentación.


In [ ]:
model_brain, model_isq, resolved_target_layer, detected_isq_outputs_logits = load_segmentation_models(
    MODEL_ISQ_PATH,
    MODEL_BRAIN_PATH,
    use_brain_mask=USE_BRAIN_MASK,
    target_layer_name=TARGET_LAYER_NAME,
)

case_dirs = sort_case_dirs(DATASET_DIR)
if not case_dirs:
    raise FileNotFoundError(f"No CASE_* directories were found under {DATASET_DIR}")

print(f"Loaded {len(case_dirs)} cases")
ISQ_MODEL_OUTPUTS_LOGITS = bool(detected_isq_outputs_logits)

print(f"Resolved ischemia target layer: {resolved_target_layer}")
print(f"Ischemia model outputs logits: {ISQ_MODEL_OUTPUTS_LOGITS}")


## 8. Pruebas rápidas de funcionamiento

Antes del análisis completo se evalúa un corte con lesión y un corte sin lesión. Esta comprobación temprana ayuda a detectar problemas de forma, mapas constantes, curvas mal normalizadas o ausencia de una región predicha que pueda explicarse.


In [ ]:
# ============================================================
# Smoke tests
# ============================================================

lesion_example, non_lesion_example = collect_smoke_examples(DATASET_DIR)
smoke_examples = [("lesion", lesion_example), ("non_lesion", non_lesion_example)]
smoke_rows = []

for example_name, example in smoke_examples:
    if example is None:
        raise RuntimeError(f"Smoke test could not find a {example_name} example in {DATASET_DIR}")

    prediction = predict_ischemia_slice(
        example["slice_img"],
        model_brain=model_brain,
        model_isq=model_isq,
        use_brain_mask=USE_BRAIN_MASK,
    )
    xai_bundle = compute_xai_bundle(
        model=model_isq,
        input_tensor=prediction["isq_input_tensor"],
        roi_mask=prediction["roi_mask"],
        target_layer_name=resolved_target_layer,
        baseline_image=prediction["baseline_image"],
        ig_steps=SMOKE_TEST_IG_STEPS,
        curve_steps=SMOKE_TEST_CURVE_STEPS,
        occlusion_patch=SMOKE_TEST_OCCLUSION_PATCH,
        occlusion_stride=SMOKE_TEST_OCCLUSION_STRIDE,
        include_occlusion=True,
        faithfulness_methods=SMOKE_TEST_CURVE_METHODS,
    )

    for map_name, attribution_map in xai_bundle["attribution_maps"].items():
        assert_normalized_map(f"{example_name}:{map_name}", attribution_map)
    assert_normalized_map(f"{example_name}:occlusion", xai_bundle["occlusion_map"])

    for method_name in SMOKE_TEST_CURVE_METHODS:
        faithfulness = xai_bundle["faithfulness_by_method"][method_name]
        for curve_name in ("deletion", "insertion", "morf", "lerf"):
            scores = faithfulness[curve_name]["scores"]
            fractions = faithfulness[curve_name]["fractions"]
            if len(scores) != SMOKE_TEST_CURVE_STEPS + 1 or len(fractions) != SMOKE_TEST_CURVE_STEPS + 1:
                raise AssertionError(f"Unexpected number of curve points for {example_name}:{method_name}:{curve_name}")
            if prediction["has_explainable_target"] and not np.isfinite(scores).all():
                raise AssertionError(f"Non-finite scores in {example_name}:{method_name}:{curve_name}")
        if prediction["has_explainable_target"]:
            assert_faithfulness_curve_sanity(
                faithfulness,
                context=f"smoke:{example_name}:{method_name}",
            )

    gradcam_faithfulness = xai_bundle["faithfulness_by_method"][SMOKE_TEST_CURVE_METHODS[0]]
    smoke_rows.append({
        "example_type": example_name,
        "case": example["case"],
        "slice_idx": example["slice_idx"],
        "gt_pixels": int(example["gt_mask"].sum()),
        "pred_pixels": int(prediction["isq_pred_mask"].sum()),
        "roi_strategy": prediction["roi_strategy"],
        "has_explainable_target": bool(prediction["has_explainable_target"]),
        "target_layer": resolved_target_layer,
        "gradcam_min": float(xai_bundle["attribution_maps"]["gradcam"].min()),
        "gradcam_max": float(xai_bundle["attribution_maps"]["gradcam"].max()),
        "deletion_points": len(gradcam_faithfulness["deletion"]["scores"]),
        "insertion_delta": float(
            gradcam_faithfulness["insertion"]["scores"][-1]
            - gradcam_faithfulness["insertion"]["scores"][0]
        ),
        "deletion_delta": float(
            gradcam_faithfulness["deletion"]["scores"][0]
            - gradcam_faithfulness["deletion"]["scores"][-1]
        ),
    })

    del prediction
    del xai_bundle
    gc.collect()

smoke_df = pd.DataFrame(smoke_rows)
print("Smoke tests passed")
print("This smoke test uses reduced-memory settings on purpose.")
smoke_df


## 9. Análisis completo del conjunto de datos

En esta fase se procesan todos los casos y cortes disponibles. Para cada corte se calculan la predicción de isquemia, las métricas frente a la máscara manual, el tipo de resultado respecto a presencia de lesión y los mapas XAI.

La clasificación TP, FP, FN o TN se usa como apoyo para seleccionar ejemplos cualitativos y discutir el comportamiento del modelo, especialmente en cortes sin lesión manual o con predicciones espurias.


In [ ]:
# ============================================================
# Full dataset analysis
# ============================================================

slice_records = []
method_metric_records = []
curve_records = []

for case_dir in case_dirs:
    nifti_path = case_dir / "Seq No.nii"
    if not nifti_path.exists():
        print(f"Skipping {case_dir.name}: missing Seq No.nii")
        continue

    nii, volume = load_nifti_volume(nifti_path)
    total_slices = int(volume.shape[2])
    lesion_masks = load_union_mask_dict(case_dir / "isquemia", volume.shape)

    print(f"Processing {case_dir.name}: {total_slices} slices")

    for slice_idx in range(total_slices):
        slice_img = volume[:, :, slice_idx]
        gt_mask_raw = lesion_masks.get(slice_idx, np.zeros_like(slice_img, dtype=np.uint8))
        gt_mask = resize_mask(gt_mask_raw, target_size=IMG_TARGET)

        prediction = predict_ischemia_slice(
            slice_img,
            model_brain=model_brain,
            model_isq=model_isq,
            use_brain_mask=USE_BRAIN_MASK,
        )
        xai_bundle = compute_xai_bundle(
            model=model_isq,
            input_tensor=prediction["isq_input_tensor"],
            roi_mask=prediction["roi_mask"],
            target_layer_name=resolved_target_layer,
            baseline_image=prediction["baseline_image"],
            ig_steps=IG_STEPS,
            curve_steps=CURVE_STEPS,
            occlusion_patch=OCCLUSION_PATCH,
            occlusion_stride=OCCLUSION_STRIDE,
            include_occlusion=True,
            faithfulness_methods=None,
        )

        slice_uid = f"{case_dir.name}__slice_{slice_idx:04d}"
        edge_label = get_edge_position_label(slice_idx, total_slices)

        gt_has_lesion = bool(gt_mask.sum() > 0)
        pred_has_lesion = bool(prediction["isq_pred_mask"].sum() > 0)
        if gt_has_lesion and pred_has_lesion:
            prediction_outcome = "TP"
        elif (not gt_has_lesion) and pred_has_lesion:
            prediction_outcome = "FP"
        elif gt_has_lesion and (not pred_has_lesion):
            prediction_outcome = "FN"
        else:
            prediction_outcome = "TN"

        slice_record = {
            "slice_uid": slice_uid,
            "case": case_dir.name,
            "slice_idx_original": int(slice_idx),
            "total_slices_case": total_slices,
            "edge_position_label": edge_label,
            "is_edge_first_last3": bool(edge_label is not None),
            "is_lesion_gt": gt_has_lesion,
            "is_non_lesion_gt": not gt_has_lesion,
            "gt_lesion_pixels": int(gt_mask.sum()),
            "pred_lesion_pixels": int(prediction["isq_pred_mask"].sum()),
            "brain_mask_used": bool(USE_BRAIN_MASK),
            "target_mask_strategy_used": prediction["roi_strategy"],
            "has_explainable_target": bool(prediction["has_explainable_target"]),
            "prediction_outcome": prediction_outcome,
            "base_score": float(prediction["base_score"]),
            "isq_dice": float(dice_numpy(gt_mask, prediction["isq_pred_mask"])),
            "isq_iou": float(iou_numpy(gt_mask, prediction["isq_pred_mask"])),
            "target_layer_name_used": resolved_target_layer,
            "selected_for_export": False,
            "selected_export_groups": "",
            "selection_reasons": "",
            "figure_paths": "",
        }
        slice_records.append(slice_record)

        for method_name, faithfulness in xai_bundle["faithfulness_by_method"].items():
            method_metric_records.append({
                "slice_uid": slice_uid,
                "case": case_dir.name,
                "slice_idx_original": int(slice_idx),
                "method": method_name,
                "deletion_auc": float(faithfulness["deletion"]["auc"]),
                "insertion_auc": float(faithfulness["insertion"]["auc"]),
                "morf_auc": float(faithfulness["morf"]["auc"]),
                "lerf_auc": float(faithfulness["lerf"]["auc"]),
                "aopc": float(faithfulness["aopc"]),
                "normalized_aopc": float(faithfulness["normalized_aopc"]),
                "occlusion_mean_drop": np.nan,
                "occlusion_max_drop": np.nan,
                "occlusion_sum_drop": np.nan,
                "figure_paths": "",
            })

            for curve_name in ("deletion", "insertion", "morf", "lerf"):
                curve_bundle = faithfulness[curve_name]
                for step_idx, (fraction, score_value, raw_score_value) in enumerate(zip(curve_bundle["fractions"], curve_bundle["scores"], curve_bundle["raw_scores"])):
                    curve_records.append({
                        "slice_uid": slice_uid,
                        "case": case_dir.name,
                        "slice_idx_original": int(slice_idx),
                        "method": method_name,
                        "curve_type": curve_name,
                        "step_idx": int(step_idx),
                        "fraction": float(fraction),
                        "score": float(score_value),
                        "raw_score": float(raw_score_value),
                    })

        occlusion_summary = xai_bundle["occlusion_summary"]
        method_metric_records.append({
            "slice_uid": slice_uid,
            "case": case_dir.name,
            "slice_idx_original": int(slice_idx),
            "method": "occlusion",
            "deletion_auc": np.nan,
            "insertion_auc": np.nan,
            "morf_auc": np.nan,
            "lerf_auc": np.nan,
            "aopc": np.nan,
            "normalized_aopc": np.nan,
            "occlusion_mean_drop": float(occlusion_summary["occlusion_mean_drop"]),
            "occlusion_max_drop": float(occlusion_summary["occlusion_max_drop"]),
            "occlusion_sum_drop": float(occlusion_summary["occlusion_sum_drop"]),
            "figure_paths": "",
        })

        del prediction
        del xai_bundle
        gc.collect()

    save_checkpoint_tables(slice_records, method_metric_records, curve_records)
    del nii
    del volume
    del lesion_masks
    gc.collect()

print("Heavy analysis finished. Checkpoint tables saved in:")
print(f"  - {CSV_DIR / 'slice_summary_checkpoint.csv'}")
print(f"  - {CSV_DIR / 'method_metrics_checkpoint.csv'}")
print(f"  - {CSV_DIR / 'curve_points_checkpoint.csv'}")
print("Run the next cell to reload checkpoint tables without repeating inference/XAI.")


## 10. Selección de cortes y exportación de figuras

A partir de las tablas generadas se seleccionan ejemplos representativos para la memoria. Se incluyen cortes con lesión, sin lesión y de los extremos del volumen, ya que estos grupos pueden mostrar comportamientos distintos del modelo.

Los mapas se recalculan solo para los cortes seleccionados, reduciendo el consumo de memoria durante la generación de figuras.


In [ ]:
# ============================================================
# Reload checkpoint tables, then slice selection and figure export
# ============================================================

slice_summary_df, method_metrics_df, curve_points_df = load_checkpoint_tables()
print(f"Loaded checkpoint tables for {slice_summary_df['case'].nunique()} cases and {len(slice_summary_df)} slices")
print(slice_summary_df[["case", "slice_idx_original", "is_lesion_gt", "is_edge_first_last3", "gt_lesion_pixels", "pred_lesion_pixels"]].head())

selected_slices_df = build_selected_slices(slice_summary_df)
exported_rows = []
case_lookup = {case_dir.name: case_dir for case_dir in case_dirs}
case_cache = {}

def get_case_context(case_name):
    if case_name not in case_cache:
        case_dir = case_lookup[case_name]
        _, volume = load_nifti_volume(case_dir / "Seq No.nii")
        lesion_masks = load_union_mask_dict(case_dir / "isquemia", volume.shape)
        case_cache[case_name] = {
            "volume": volume,
            "lesion_masks": lesion_masks,
        }
    return case_cache[case_name]

for _, selected_row in selected_slices_df.iterrows():
    case_name = selected_row["case"]
    slice_idx = int(selected_row["slice_idx_original"])
    context = get_case_context(case_name)

    slice_img = context["volume"][:, :, slice_idx]
    gt_mask = resize_mask(
        context["lesion_masks"].get(slice_idx, np.zeros_like(slice_img, dtype=np.uint8)),
        target_size=IMG_TARGET,
    )

    prediction = predict_ischemia_slice(
        slice_img,
        model_brain=model_brain,
        model_isq=model_isq,
        use_brain_mask=USE_BRAIN_MASK,
    )
    xai_bundle = compute_xai_bundle(
        model=model_isq,
        input_tensor=prediction["isq_input_tensor"],
        roi_mask=prediction["roi_mask"],
        target_layer_name=resolved_target_layer,
        baseline_image=prediction["baseline_image"],
        ig_steps=IG_STEPS,
        curve_steps=CURVE_STEPS,
        occlusion_patch=OCCLUSION_PATCH,
        occlusion_stride=OCCLUSION_STRIDE,
        include_occlusion=True,
        faithfulness_methods=None,
    )

    artifact = {
        "image": prediction["resized_img"],
        "gt_mask": gt_mask,
        "gradcam": xai_bundle["attribution_maps"]["gradcam"],
        "gradcampp": xai_bundle["attribution_maps"]["gradcampp"],
        "ig": xai_bundle["attribution_maps"]["ig"],
        "occlusion": xai_bundle["occlusion_map"],
        "faithfulness_by_method": xai_bundle["faithfulness_by_method"],
        "meta": {
            "case": case_name,
            "slice_idx_original": slice_idx,
            "gt_lesion_pixels": int(gt_mask.sum()),
            "pred_lesion_pixels": int(prediction["isq_pred_mask"].sum()),
            "target_mask_strategy_used": prediction["roi_strategy"],
            "has_explainable_target": bool(prediction["has_explainable_target"]),
        },
    }

    output_path = FIGURES_DIR / selected_row["selection_group"] / make_export_filename(
        case_name,
        slice_idx,
    )
    render_slice_montage(artifact, output_path)
    exported_rows.append({
        "slice_uid": selected_row["slice_uid"],
        "case": case_name,
        "slice_idx_original": slice_idx,
        "selection_group": selected_row["selection_group"],
        "selection_reason": selected_row["selection_reason"],
        "figure_path": str(output_path),
    })
    pd.DataFrame(exported_rows).to_csv(CSV_DIR / "selected_slices.csv", index=False)

    del prediction
    del xai_bundle
    del artifact
    gc.collect()

selected_slices_df = pd.DataFrame(exported_rows)

if not selected_slices_df.empty:
    group_map = selected_slices_df.groupby("slice_uid")["selection_group"].agg(lambda values: "|".join(sorted(set(values))))
    reason_map = selected_slices_df.groupby("slice_uid")["selection_reason"].agg(lambda values: "|".join(values))
    figure_map = selected_slices_df.groupby("slice_uid")["figure_path"].agg(lambda values: "|".join(values))

    slice_summary_df.loc[slice_summary_df["slice_uid"].isin(group_map.index), "selected_for_export"] = True
    slice_summary_df["selected_export_groups"] = slice_summary_df["slice_uid"].map(group_map).fillna("")
    slice_summary_df["selection_reasons"] = slice_summary_df["slice_uid"].map(reason_map).fillna("")
    slice_summary_df["figure_paths"] = slice_summary_df["slice_uid"].map(figure_map).fillna("")
    method_metrics_df["figure_paths"] = method_metrics_df["slice_uid"].map(figure_map).fillna("")

print(f"Exported {len(selected_slices_df)} montage files")
selected_slices_df.head()


## 11. Exportación final e interpretación

Esta última fase consolida las tablas CSV y resume el número de casos, cortes y figuras generadas. Las salidas permiten separar análisis cuantitativo, fidelidad de las explicaciones y revisión visual.

Notas de interpretación para la memoria:

- La explicación se calcula sobre la lesión predicha, porque se pretende inspeccionar la decisión efectivamente producida por el modelo.
- La máscara manual se usa para evaluar concordancia y para clasificar cortes como TP, FP, FN o TN, no como objetivo directo de la explicación.
- Los cortes sin lesión predicha no tienen una región de primer plano explicable; sus métricas de fidelidad se exportan como `NaN` y deben excluirse de promedios globales de XAI.
- La unión de máscaras por corte evita contar varias anotaciones de una misma lesión o de lesiones múltiples como muestras independientes.
- Un mapa de atribución compatible con la zona lesionada sugiere coherencia espacial, pero no demuestra causalidad ni garantiza validez clínica.
- Grad-CAM++ se usa con una aproximación orientada a memoria, documentada porque la formulación exacta puede ser inviable en algunos entornos de ejecución.


In [ ]:
# ============================================================
# Final CSV exports
# ============================================================

slice_summary_path = CSV_DIR / "slice_summary.csv"
method_metrics_path = CSV_DIR / "method_metrics_long.csv"
curve_points_path = CSV_DIR / "curve_points_long.csv"
curve_points_checkpoint_path = CSV_DIR / "curve_points_checkpoint.csv"
group_summary_path = CSV_DIR / "group_summary.csv"
selected_slices_path = CSV_DIR / "selected_slices.csv"

if "slice_summary_df" not in globals() or "method_metrics_df" not in globals() or "curve_points_df" not in globals():
    slice_summary_df, method_metrics_df, curve_points_df = load_checkpoint_tables()

if "selected_slices_df" not in globals():
    if selected_slices_path.exists():
        selected_slices_df = pd.read_csv(selected_slices_path)
    else:
        selected_slices_df = pd.DataFrame(columns=["slice_uid", "case", "slice_idx_original", "selection_group", "selection_reason", "figure_path"])

group_summary_df = build_group_summary(method_metrics_df, slice_summary_df)

slice_summary_df.to_csv(slice_summary_path, index=False)
method_metrics_df.to_csv(method_metrics_path, index=False)
if curve_points_df.empty and curve_points_checkpoint_path.exists():
    curve_points_df = pd.read_csv(curve_points_checkpoint_path)
curve_points_df.to_csv(curve_points_path, index=False)
group_summary_df.to_csv(group_summary_path, index=False)

if selected_slices_df.empty:
    pd.DataFrame(columns=["slice_uid", "case", "slice_idx_original", "selection_group", "selection_reason", "figure_path"]).to_csv(selected_slices_path, index=False)
else:
    selected_slices_df.to_csv(selected_slices_path, index=False)

print("Saved outputs:")
print(f"  - {slice_summary_path}")
print(f"  - {method_metrics_path}")
print(f"  - {curve_points_path}")
print(f"  - {group_summary_path}")
print(f"  - {selected_slices_path}")

print("\nSummary counts")
print(f"  Cases: {slice_summary_df['case'].nunique()}")
print(f"  Slices: {len(slice_summary_df)}")
print(f"  Lesion slices: {int(slice_summary_df['is_lesion_gt'].sum())}")
print(f"  Non-lesion slices: {int(slice_summary_df['is_non_lesion_gt'].sum())}")
print(f"  Edge first/last 3 slices: {int(slice_summary_df['is_edge_first_last3'].sum())}")
if "prediction_outcome" in slice_summary_df.columns:
    print("  Prediction outcomes:")
    for outcome, count in slice_summary_df["prediction_outcome"].value_counts(dropna=False).sort_index().items():
        print(f"    - {outcome}: {int(count)}")
if "has_explainable_target" in slice_summary_df.columns:
    print(f"  Slices included in quantitative XAI metrics: {int(slice_summary_df['has_explainable_target'].sum())}")
print(f"  Exported montages: {len(selected_slices_df)}")
